# Notebook 15B — Fannie Mae EDA and population comparison

We examine the candidate Fannie population before developing companion models.
The main work is numerical and categorical EDA, missingness, outlier diagnostics,
correlations and comparisons with Freddie development and 2018 Q1 populations.
The existing 29-predictor models and all previous results remain unchanged.

**Run:** open the repository in VS Code, select the Python `.venv` kernel and run
the cells in order. Use the processed files already created by Notebooks 03, 06,
10, 14G and 15. No additional package is required beyond the project dependencies
and SciPy (installed with scikit-learn). Set `FANNIE_RUN` if more than one
Notebook 15 output folder exists. Missing or ambiguous inputs stop with a message.

**Scope:** Fannie actual 2018 Q1 originations contained in one acquisition file,
not the complete Fannie origination cohort. Freddie is an annual sample.
Sampled originations and eligible modeling loans are reported separately.
All EDA uses values before imputation. FICO is a sensitivity proxy.
There are no feature-versus-target analyses, model fitting or predictions here.

**Stages:** verify inputs → reconcile quality → numerical EDA → categorical EDA
→ correlations and population comparisons → distribution distances → readiness.
The final cell saves aggregate tables, actual analysis figures and a review ZIP.


## 1. Set paths and verify the candidate dataset

Select one complete Notebook 15 run, then verify its fingerprint, 14G proposal,
original Freddie feature contract and saved aggregate counts. We do not choose
the newest run silently. The optional raw audit is off by default because the
processed dataset supports most EDA. Leave approval notes blank where evidence
has not been reviewed; completing this notebook does not authorize scoring.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import importlib
import json
import platform
import sys
import zipfile

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.stats import wasserstein_distance
from IPython.display import display

ROOT = (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()).resolve()
FREDDIE = ROOT / 'data/processed/Freddie_Mac'
FANNIE = ROOT / 'data/processed/Fannie_Mae'
BASE = FREDDIE / 'model_baselines_2015_2017'

FANNIE_RUN = None  # Example: FANNIE / 'notebook_15_external_dataset_construction/20261124T164838809452Z'
REVIEW_14G = None  # Exact reviewed ZIP, only needed when discovery is ambiguous.
RUN_TARGETED_RAW_AUDIT = False  # Optional: one full sequential scan, retaining only needed evidence.
RAW_FANNIE = ROOT / 'data/raw/Fannie_Mae/2018Q1/2018Q1.csv'
RELEASE_REVIEW_NOTE = ''  # Record the release identifier, matching glossary and reviewed evidence.
TARGET_REVIEW_NOTE = ''  # Record the remaining cross-agency terminal-event interpretation.

EXPECTED_N15 = dict(total=260395, primary=260244, strict=260043,
                    primary_events=1960, strict_events=1947,
                    conflicts=56, repurchases=200)
EXPECTED_FREDDIE_ANNUAL = 50000
EXPECTED_FREDDIE_DEVELOPMENT = 143598
REVIEWED_FEATURE_BLOB = '7da952013fd70095b001483f94256b4cf0a55e7d'
SOURCE_COMMIT = '0e7e1e3741ead869915463bc8dfc824bed02a85a'
KEYS = ['SOURCE YEAR', 'LOAN IDENTIFIER']
LABELS = ['TARGET_90_PLUS', 'TARGET_COMPOSITE_CREDIT_EVENT']

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def select_unique(paths, description):
    paths = sorted(set(Path(p).resolve() for p in paths))
    require(bool(paths), f'No {description} found. Check the configuration paths.')
    require(len(paths) == 1, f'Multiple {description} found; select one explicitly: {paths}')
    return paths[0]

run_paths = [Path(FANNIE_RUN)] if FANNIE_RUN else [
    p.parent for p in (FANNIE / 'notebook_15_external_dataset_construction').glob('*/metadata.json')]
N15 = select_unique(run_paths, 'Notebook 15 run folders')
paths = {
    'candidate': N15 / 'fannie_2018q1_candidate_dataset.parquet',
    'metadata_15': N15 / 'metadata.json',
    'summary_15': N15 / 'target_summary.csv',
    'profile_15': N15 / 'feature_profile.csv',
    'quality_15': N15 / 'quality_checks.csv',
    'review_15': N15 / 'notebook_15_external_dataset_review.zip',
    'manifest': BASE / 'model_manifest.json',
    'frozen_contract': BASE / 'freddie_feature_contract.json',
    'modeling': FREDDIE / 'freddie_modeling_dataset_2015_2023_24m.parquet',
    'feature_source': ROOT / 'src/freddie_features.py',
}
paths.update({f'origination_{y}': FREDDIE / f'origination_{y}_clean.parquet'
              for y in [2015, 2016, 2017, 2018]})
for name, path in paths.items():
    require(path.is_file(), f'Missing {name}: {path}')

meta = json.loads(paths['metadata_15'].read_text(encoding='utf-8'))
manifest = json.loads(paths['manifest'].read_text(encoding='utf-8'))
contract = json.loads(paths['frozen_contract'].read_text(encoding='utf-8'))
hashes = {name: sha256(path) for name, path in paths.items()}
require(hashes['candidate'] == meta['candidate_dataset_sha256'], 'Candidate dataset fingerprint differs.')
require(hashes['frozen_contract'] == meta['frozen_contract_sha256'] == manifest['feature_contract_sha256'],
        'Original Freddie feature contract differs.')
require(hashes['modeling'] == manifest['input_file_sha256'], 'Freddie modeling file differs from baseline input.')
require(manifest['horizon_months'] == 24 and manifest['development_years'] == [2015, 2016, 2017],
        'Original development design differs.')
require(manifest['target'] == 'TARGET_COMPOSITE_CREDIT_EVENT', 'Original target differs.')
require(meta['physical_rows'] == 23563399 and meta['unique_loans'] == 458830, 'Notebook 15 source totals differ.')
require(meta['source_bytes'] == 6928164686, 'Notebook 15 source size differs.')
require(meta['sample_reconciliation'] == dict(loans=4749, rows=245552, primary=4491,
        strict=4488, primary_events=33, strict_events=33), 'Diagnostic sample reconciliation differs.')
require(meta['models_scored'] is False, 'Expected the unscored candidate dataset.')
with zipfile.ZipFile(paths['review_15']) as archive:
    for filename in ['metadata.json', 'target_summary.csv', 'feature_profile.csv', 'quality_checks.csv']:
        require(archive.read(filename) == (N15 / filename).read_bytes(), f'Notebook 15 ZIP disagrees: {filename}')

source_bytes = paths['feature_source'].read_bytes().replace(b'\r\n', b'\n')
blob = hashlib.sha1(b'blob ' + str(len(source_bytes)).encode() + b'\0' + source_bytes).hexdigest()
require(blob == REVIEWED_FEATURE_BLOB, 'Feature source changed; reconcile it before using this notebook.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
features_module = importlib.import_module('src.freddie_features')
require(Path(features_module.__file__).resolve() == paths['feature_source'].resolve(), 'Imported another repository.')
features_module = importlib.reload(features_module)
require(manifest['predictors'] == features_module.PREDICTOR_COLUMNS, 'Original predictor order differs.')
require(contract['training_years'] == [2015, 2016, 2017], 'Frozen preprocessing reference differs.')

review_paths = [Path(REVIEW_14G)] if REVIEW_14G else list(FANNIE.rglob('notebook_14g_final_compatibility_review.zip'))
matching_reviews = [p for p in review_paths if p.is_file() and sha256(p) == meta['review_14g_sha256']]
require(bool(matching_reviews), 'Cannot locate the exact 14G review ZIP identified by Notebook 15.')
paths['review_14g'] = sorted(matching_reviews)[0]  # Byte-identical copies are equivalent.
hashes['review_14g'] = sha256(paths['review_14g'])
with zipfile.ZipFile(paths['review_14g']) as archive:
    proposal = json.loads(archive.read('candidate_contract.json'))
PRIMARY = proposal['original_predictors_retained'] + proposal['additional_candidate']
SENSITIVITY = PRIMARY + ['FICO_SCORE', 'FICO_SCORE_MISSING']
require(len(PRIMARY) == len(set(PRIMARY)) == 22 and len(SENSITIVITY) == 24, 'Candidate contract differs.')
require(PRIMARY == meta['primary_features'] and SENSITIVITY == meta['fico_sensitivity_features'],
        'Notebook 14G and 15 predictor contracts disagree.')
excluded = {'FICO_SCORE', 'FICO_SCORE_MISSING', 'FIRST_TIME_HOMEBUYER', 'LOAN_PURPOSE',
            'SUPER_CONFORMING_FLAG', 'SPECIAL_ELIGIBILITY_PROGRAM', 'HARP_INDICATOR', 'INTEREST_ONLY_INDICATOR'}
require(PRIMARY == [c for c in features_module.PREDICTOR_COLUMNS if c not in excluded]
        + ['PURCHASE_REFINANCE_GROUP'], 'Candidate specification differs from the reviewed 22-feature proposal.')
require(proposal['frozen_contract_sha256'] == hashes['frozen_contract'] and not proposal['models_scored'],
        'Notebook 14G provenance differs.')
signatures = {name: (p.stat().st_size, p.stat().st_mtime_ns) for name, p in paths.items()}
print('Verified the candidate file, review bundles and original Freddie contracts. No model was loaded.')
print('Notebook 15 run:', N15)


In [ ]:
fannie = pd.read_parquet(paths['candidate'])
audit_columns = ['loan_id', 'orig', 'first_payment', 'report', 'primary', 'strict', 'reason',
                 'missing_months', 'unknown_active', 'repurchase', 'terminal_timing_disagreements',
                 'origination_field_conflict_count', 'MI_MAPPING_INCONSISTENT', 'ORIGINATION_COHORT']
target_columns = [f'TARGET_{p}_{t}' for p in ['PRIMARY', 'STRICT']
                  for t in ['COMPOSITE', 'NUMERIC_90_PLUS']]
require(set(audit_columns + SENSITIVITY + target_columns).issubset(fannie.columns), 'Candidate columns missing.')
require(len(fannie) == EXPECTED_N15['total'] and fannie.loan_id.notna().all()
        and fannie.loan_id.is_unique, 'Fannie loan count or keys differ.')
for c in ['primary', 'strict', 'unknown_active', 'repurchase', 'MI_MAPPING_INCONSISTENT']:
    require(fannie[c].notna().all() and fannie[c].isin([True, False]).all(), f'Invalid audit flag: {c}')
require(fannie.ORIGINATION_COHORT.eq('2018Q1').all(), 'Fannie cohort differs.')
orig_dates = pd.to_datetime(fannie.orig.astype('string'), format='%m%Y', errors='raise')
require(orig_dates.dt.year.eq(2018).all() and orig_dates.dt.quarter.eq(1).all(), 'Actual origination dates differ.')
require((~fannie.strict | fannie.primary).all(), 'Strict must be a subset of primary.')
for name in ['ORIGINAL_DTI', 'ORIGINAL_LTV', 'FICO_SCORE', 'MI_PERCENT']:
    require(fannie[name + '_MISSING'].eq(fannie[name].isna().astype(int)).all(),
            f'Fannie missingness indicator differs from its retained predictor: {name}')
for c in ['missing_months', 'terminal_timing_disagreements', 'origination_field_conflict_count']:
    v = pd.to_numeric(fannie[c], errors='raise')
    require(v.notna().all() and v.ge(0).all() and v.eq(np.floor(v)).all(), f'Invalid audit counts: {c}')

summaries = []
for p in ['primary', 'strict']:
    e = fannie[p]
    for t in ['COMPOSITE', 'NUMERIC_90_PLUS']:
        y = fannie[f'TARGET_{p.upper()}_{t}']
        require(y.loc[e].notna().all() and y.loc[e].isin([0, 1]).all()
                and y.loc[~e].isna().all(), f'Eligibility/label mismatch: {p}/{t}')
    require((fannie.loc[e, f'TARGET_{p.upper()}_NUMERIC_90_PLUS'] <=
             fannie.loc[e, f'TARGET_{p.upper()}_COMPOSITE']).all(), 'Numeric outcome exceeds composite.')
    n = int(e.sum())
    events = int(fannie.loc[e, f'TARGET_{p.upper()}_COMPOSITE'].sum())
    require(n == EXPECTED_N15[p] and events == EXPECTED_N15[f'{p}_events'], 'Notebook 15 totals changed.')
    summaries.append(dict(policy=p, total=len(fannie), eligible=n, ineligible=len(fannie)-n,
                          composite=events, numeric_serious=int(fannie.loc[e, f'TARGET_{p.upper()}_NUMERIC_90_PLUS'].sum()),
                          event_rate=events/n))
target_summary = pd.DataFrame(summaries)
saved_summary = pd.read_csv(paths['summary_15']).set_index('policy')
for row in target_summary.to_dict('records'):
    for c in ['total', 'eligible', 'ineligible', 'composite', 'numeric_serious']:
        require(row[c] == saved_summary.loc[row['policy'], c], f'Saved summary differs: {c}')
    require(np.isclose(row['event_rate'], saved_summary.loc[row['policy'], 'event_rate']), 'Event rate differs.')
pd.testing.assert_series_equal(fannie.loc[fannie.strict, 'TARGET_PRIMARY_COMPOSITE'],
        fannie.loc[fannie.strict, 'TARGET_STRICT_COMPOSITE'], check_names=False)
reproduced_quality = {
    'mi_mapping_inconsistent': int(fannie.MI_MAPPING_INCONSISTENT.sum()),
    'origination_field_conflicts': int(fannie.origination_field_conflict_count.gt(0).sum()),
    'terminal_window_disagreement': int(fannie.terminal_timing_disagreements.gt(0).sum()),
    'unknown_active': int(fannie.unknown_active.sum()), 'repurchase': int(fannie.repurchase.sum())}
require(reproduced_quality['origination_field_conflicts'] == EXPECTED_N15['conflicts']
        and reproduced_quality['repurchase'] == EXPECTED_N15['repurchases'], 'Quality totals differ.')
require(reproduced_quality == pd.read_csv(paths['quality_15']).set_index('check').affected_loans.to_dict(),
        'Quality checks do not reproduce Notebook 15.')
saved_profile = pd.read_csv(paths['profile_15']).set_index('predictor')
for c in SENSITIVITY:
    require(int(fannie.loc[fannie.primary, c].isna().sum()) == saved_profile.loc[c, 'missing']
            and int(fannie.loc[fannie.primary, c].nunique()) == saved_profile.loc[c, 'distinct'],
            f'Saved feature profile differs: {c}')
display(target_summary)
display(pd.DataFrame(reproduced_quality.items(), columns=['check', 'affected_loans']))


## 2. Prepare Freddie populations before imputation

Use the reviewed feature transformations. For the full origination sample we
mirror the label-free adapter used in Notebook 12, then check every feature
against the original builder on eligible loans. Integer category spellings
such as `1.0` and `1` are harmonized for units and loan term only.

Purchase/refinance is provisional: Freddie P → purchase, C/N → refinance;
other codes stay unmapped and are counted. No category is inferred from outcomes.
The development reference includes all existing loans; no new product restriction
or exclusion is introduced. Product coverage differences remain review issues.


In [ ]:
def normalize_freddie(frame, name):
    frame = frame.copy()
    required = KEYS + ['ORIGINATION YEAR', 'ORIGINATION QUARTER', 'ORIGINATION COHORT']
    require(set(required).issubset(frame.columns) and not frame[required].isna().any().any(),
            f'{name}: missing keys/cohorts.')
    parts = frame['LOAN IDENTIFIER'].astype('string').str.extract(r'^[FA]([0-9]{2})Q([1-4])[0-9]{7}$')
    require(not parts.isna().any().any(), f'{name}: invalid Freddie identifiers.')
    year = pd.to_numeric(frame['ORIGINATION YEAR'], errors='raise')
    quarter = pd.to_numeric(frame['ORIGINATION QUARTER'].astype('string').str.removeprefix('Q'), errors='raise')
    require(year.eq(2000 + parts[0].astype(int)).all() and quarter.eq(parts[1].astype(int)).all(),
            f'{name}: identifier and cohort disagree.')
    require(pd.to_numeric(frame['SOURCE YEAR'], errors='raise').eq(year).all(), f'{name}: source year differs.')
    frame['SOURCE YEAR'] = year.astype(int)
    expected_cohort = year.astype(int).astype(str) + 'Q' + quarter.astype(int).astype(str)
    require(frame['ORIGINATION COHORT'].astype('string').eq(expected_cohort).all(),
        f'{name}: saved cohort text differs.')
    frame['ORIGINATION COHORT'] = expected_cohort
    require(not frame.duplicated(KEYS).any(), f'{name}: duplicate keys.')
    return frame

origin_columns = [c for c in features_module.REQUIRED_SOURCE_COLUMNS if c not in LABELS + ['SOURCE YEAR']]
parts = []
for year in [2015, 2016, 2017, 2018]:
    part = pd.read_parquet(paths[f'origination_{year}'], columns=origin_columns)
    part['SOURCE YEAR'] = year
    part = normalize_freddie(part, f'originations {year}')
    require(len(part) == EXPECTED_FREDDIE_ANNUAL, f'{year}: annual sample size differs.')
    parts.append(part)
originations = pd.concat(parts, ignore_index=True)
require(not originations.duplicated(KEYS).any(), 'Duplicate combined Freddie keys.')
modeling = normalize_freddie(pd.read_parquet(paths['modeling'],
    columns=features_module.REQUIRED_SOURCE_COLUMNS + ['HORIZON_MONTHS']), 'modeling')
require(modeling.HORIZON_MONTHS.eq(24).all(), 'Freddie target horizon differs.')
modeling = modeling.loc[modeling['SOURCE YEAR'].between(2015, 2018)].copy()
require(modeling[LABELS].isin([0, 1]).all().all(), 'Invalid Freddie labels.')
dev = modeling['SOURCE YEAR'].between(2015, 2017)
require(int(dev.sum()) == EXPECTED_FREDDIE_DEVELOPMENT == manifest['development_rows'], 'Development count differs.')
require(int(modeling.loc[dev, LABELS[1]].sum()) == manifest['development_events'], 'Development events differ.')
origin_index = originations.set_index(KEYS)
model_index = modeling.set_index(KEYS)
require(model_index.index.isin(origin_index.index).all(), 'Eligible Freddie loans absent from origination sample.')
for c in origin_columns:
    if c != 'LOAN IDENTIFIER':
        pd.testing.assert_series_equal(origin_index.loc[model_index.index, c], model_index[c],
                                       check_dtype=False, check_names=False, check_exact=True)

def origination_predictors(source):
    require(not set(LABELS).intersection(source.columns), 'Unexpected outcomes in origination adapter.')
    out = source[KEYS + ['ORIGINATION YEAR', 'ORIGINATION QUARTER', 'ORIGINATION COHORT']].copy()
    def numeric(c):
        v = pd.to_numeric(source[c], errors='raise')
        require(np.isfinite(v.dropna()).all(), f'Nonfinite source: {c}')
        return v
    fico = numeric('CLASSIC FICO_CLEAN')
    mi = numeric('MORTGAGE INSURANCE PERCENTAGE (MI %)_CLEAN')
    dti = numeric('ORIGINAL DEBT-TO-INCOME (DTI) RATIO_CLEAN')
    ltv = numeric('ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN')
    cltv = numeric('ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN')
    upb = numeric('ORIGINAL UPB')
    for name, v in [('FICO_SCORE', fico), ('MI_PERCENT', mi), ('ORIGINAL_DTI', dti),
                    ('ORIGINAL_LTV', ltv), ('ORIGINAL_INTEREST_RATE', numeric('ORIGINAL INTEREST RATE'))]:
        out[name] = v
    out['LOG_ORIGINAL_UPB'] = np.log1p(upb.where(upb.ge(0)))
    out['CLTV_MINUS_LTV'] = (cltv-ltv).clip(lower=0)
    out['HAS_MORTGAGE_INSURANCE'] = mi.gt(0).astype('int8')
    out['HAS_SUBORDINATE_FINANCING'] = cltv.gt(ltv).astype('int8')
    out['HIGH_LTV_GT_80'] = ltv.gt(80).astype('int8')
    for name, v in [('FICO_SCORE', fico), ('MI_PERCENT', mi), ('ORIGINAL_DTI', dti),
                    ('ORIGINAL_LTV', ltv), ('CLTV', cltv)]:
        out[name + '_MISSING'] = v.isna().astype('int8')
    mapping = {
        'FIRST_TIME_HOMEBUYER': 'FIRST TIME HOMEBUYER INDICATOR_CLEAN', 'NUMBER_OF_UNITS': 'NUMBER OF UNITS_CLEAN',
        'OCCUPANCY_STATUS': 'OCCUPANCY STATUS_CLEAN', 'CHANNEL': 'CHANNEL_CLEAN',
        'AMORTIZATION_TYPE': 'AMORTIZATION TYPE', 'PROPERTY_STATE': 'PROPERTY STATE',
        'PROPERTY_TYPE': 'PROPERTY TYPE_CLEAN', 'LOAN_PURPOSE': 'LOAN PURPOSE_CLEAN',
        'ORIGINAL_LOAN_TERM': 'ORIGINAL LOAN TERM', 'NUMBER_OF_BORROWERS_GROUP': 'NUMBER OF BORROWERS GROUP_CLEAN',
        'SUPER_CONFORMING_FLAG': 'SUPER CONFORMING FLAG', 'SPECIAL_ELIGIBILITY_PROGRAM': 'SPECIAL ELIGIBILITY PROGRAM',
        'HARP_INDICATOR': 'HARP INDICATOR', 'INTEREST_ONLY_INDICATOR': 'INTEREST ONLY (I/O) INDICATOR'}
    for name, c in mapping.items():
        out[name] = source[c].astype('string').str.strip()
    return out

full_freddie = origination_predictors(originations)
original_engineered = features_module.build_engineered_features(modeling)
pd.testing.assert_frame_equal(
    full_freddie.set_index(KEYS).loc[model_index.index, features_module.PREDICTOR_COLUMNS],
    original_engineered.set_index(KEYS)[features_module.PREDICTOR_COLUMNS], check_exact=True)

def integer_category(series):
    text = series.astype('string').str.strip()
    require(text.dropna().str.fullmatch(r'[0-9]+(?:[.]0+)?').all(), 'Unexpected integer category representation.')
    return text.str.replace(r'[.]0+$', '', regex=True)

full_freddie['PURCHASE_REFINANCE_GROUP'] = full_freddie.LOAN_PURPOSE.map(
    {'P': 'PURCHASE', 'C': 'REFINANCE', 'N': 'REFINANCE'})
purpose_unmapped = (full_freddie.loc[full_freddie.PURCHASE_REFINANCE_GROUP.isna()]
    .groupby(['SOURCE YEAR', 'LOAN_PURPOSE'], dropna=False).size().reset_index(name='loans'))
for c in ['NUMBER_OF_UNITS', 'ORIGINAL_LOAN_TERM']:
    full_freddie[c] = integer_category(full_freddie[c])
    fannie[c] = integer_category(fannie[c])
full_freddie['ORIGINAL_UPB_EDA'] = pd.to_numeric(originations['ORIGINAL UPB'], errors='raise').to_numpy()
full_freddie['ORIGINAL_CLTV_EDA'] = pd.to_numeric(
    originations['ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN'], errors='raise').to_numpy()
fannie['ORIGINAL_UPB_EDA'] = np.expm1(pd.to_numeric(fannie.LOG_ORIGINAL_UPB, errors='raise'))
# CLTV cannot be recovered from LTV + clipped gap when CLTV < LTV or either input was missing.
fannie['ORIGINAL_CLTV_EDA'] = np.nan
full_freddie['eligible'] = pd.MultiIndex.from_frame(full_freddie[KEYS]).isin(model_index.index)
display(purpose_unmapped)
print('Rebuilt and reconciled Freddie predictors before imputation. No preprocessing was fitted.')


## 3. Reconcile quality and eligibility

Strict eligibility adds three requirements to primary eligibility: no missing
in-window reporting month, no unknown active status and no reported in-window
repurchase. We count their overlap instead of assuming the 200 repurchases
explain all 201 additional exclusions. Conflict flags are investigated separately;
we do not drop those loans or replace their earliest values.

The optional audit below reads the raw file once. It retains earliest-snapshot
CLTV/UPB for EDA and aggregate changes for flagged loan histories. It verifies the
raw SHA-256 against Notebook 15. Without it, raw CLTV and field-level conflict
resolution remain explicitly unavailable. No raw records are exported.


In [ ]:
rules = pd.DataFrame({
    'gap_in_window': fannie.missing_months.gt(0),
    'unknown_active': fannie.unknown_active.astype(bool),
    'repurchase': fannie.repurchase.astype(bool)}, index=fannie.index)
expected_strict = fannie.primary & ~rules.any(axis=1)
require(expected_strict.eq(fannie.strict).all(), 'Stored strict eligibility does not match its stated rules.')
additional = fannie.primary & ~fannie.strict
eligibility_overlap = (rules.loc[additional].value_counts(dropna=False)
                       .rename('loans').reset_index())
require(int(eligibility_overlap.loans.sum()) == EXPECTED_N15['primary']-EXPECTED_N15['strict'],
        'Additional strict exclusions do not reconcile.')
quality_membership = []
for name, mask in {'all': pd.Series(True, index=fannie.index), 'primary': fannie.primary,
                   'strict': fannie.strict, 'primary_not_strict': additional,
                   'primary_ineligible': ~fannie.primary}.items():
    quality_membership.append(dict(population=name, loans=int(mask.sum()),
        gaps=int((mask & rules.gap_in_window).sum()),
        unknown=int((mask & rules.unknown_active).sum()),
        repurchases=int((mask & rules.repurchase).sum()),
        conflicting_origination_fields=int((mask & fannie.origination_field_conflict_count.gt(0)).sum())))
quality_membership = pd.DataFrame(quality_membership)
eligibility_reasons = fannie.groupby(['primary', 'strict', 'reason'], dropna=False).size().reset_index(name='loans')
conflict_distribution = (fannie.origination_field_conflict_count.value_counts().sort_index()
                        .rename_axis('conflicting_fields').reset_index(name='loans'))
display(quality_membership)
display(eligibility_overlap)
display(eligibility_reasons)
print('Primary-to-strict event difference:', int(fannie.loc[additional, 'TARGET_PRIMARY_COMPOSITE'].sum()))
snapshot_dates = pd.DataFrame({
    'origination': pd.to_datetime(fannie.orig.astype('string'), format='%m%Y', errors='raise'),
    'first_payment': pd.to_datetime(fannie.first_payment.astype('string'), format='%m%Y', errors='raise'),
    'first_report': pd.to_datetime(fannie.report.astype('string'), format='%m%Y', errors='raise')})
month_number = snapshot_dates.apply(lambda s: 12*s.dt.year + s.dt.month)
snapshot_lags = pd.DataFrame({
    'first_report_minus_origination_months': month_number.first_report-month_number.origination,
    'first_report_minus_first_payment_months': month_number.first_report-month_number.first_payment,
    'first_payment_minus_origination_months': month_number.first_payment-month_number.origination})
snapshot_lag_summary = snapshot_lags.describe(percentiles=[.01, .5, .95, .99]).T.reset_index(names='measure')
display(snapshot_lag_summary)

raw_audit_done = False
raw_conflicts = pd.DataFrame(columns=['field', 'flagged_loans_with_nonblank_conflict', 'changed_from_earliest'])
raw_snapshot_quality = pd.DataFrame(columns=['check', 'loans'])
if RUN_TARGETED_RAW_AUDIT:
    require(RAW_FANNIE.is_file(), f'Raw file missing: {RAW_FANNIE}')
    raw_signature = (RAW_FANNIE.stat().st_size, RAW_FANNIE.stat().st_mtime_ns)
    require(raw_signature[0] == meta['source_bytes'], 'Raw file size differs.')
    # Positions are the same provisional layout used by 14G/15, not a new release certification.
    fields = {'channel': 4, 'rate': 8, 'upb': 10, 'term': 13, 'ltv': 20, 'cltv': 21,
              'borrowers': 22, 'dti': 23, 'borrower_fico': 24, 'coborrower_fico': 25,
              'purpose': 27, 'property_type': 28, 'units': 29, 'occupancy': 30,
              'state': 31, 'mi': 34, 'amortization': 35, 'mi_type': 73}
    flagged = set(fannie.loc[fannie.origination_field_conflict_count.gt(0), 'loan_id'].astype(str))
    earliest = dict(zip(fannie.loan_id.astype(str), fannie.report.astype(str)))
    snapshots, histories = {}, {key: [] for key in flagged}
    raw_hash = hashlib.sha256()
    with RAW_FANNIE.open('rb') as stream:
        for row_number, raw in enumerate(stream, 1):
            raw_hash.update(raw)
            parts = raw.rstrip(b'\r\n').split(b'|')
            require(len(parts) == 113, f'Raw layout changed at row {row_number}.')
            key = parts[1].strip().decode('ascii')
            report = parts[2].strip().decode('ascii')
            if key in earliest and report == earliest[key]:
                require(key not in snapshots, 'Duplicate earliest reporting snapshot.')
                snapshots[key] = {name: parts[position-1].strip().decode('ascii') for name, position in fields.items()}
            if key in flagged:
                histories[key].append({name: parts[position-1].strip().decode('ascii') for name, position in fields.items()})
            if row_number % 5000000 == 0:
                print(f'Optional audit: {row_number:,} rows read')
    require(raw_hash.hexdigest() == meta['source_sha256'], 'Raw source fingerprint differs.')
    require(row_number == meta['physical_rows'] and len(snapshots) == len(fannie), 'Raw snapshot coverage differs.')
    require(raw_signature == (RAW_FANNIE.stat().st_size, RAW_FANNIE.stat().st_mtime_ns), 'Raw source changed during audit.')
    raw_first = pd.DataFrame.from_dict(snapshots, orient='index').reindex(fannie.loan_id.astype(str))
    raw_first.index = fannie.index
    cltv = pd.to_numeric(raw_first.cltv, errors='coerce')
    fannie['ORIGINAL_CLTV_EDA'] = cltv.where(cltv.between(1, 998))
    raw_upb = pd.to_numeric(raw_first.upb, errors='coerce').where(lambda s: s.gt(0))
    require(np.allclose(raw_upb, fannie.ORIGINAL_UPB_EDA, equal_nan=True), 'Raw UPB differs from saved log representation.')
    raw_rows = []
    for field in fields:
        conflict_n = changed_n = 0
        for key, history in histories.items():
            values = {r[field] for r in history if r[field]}
            conflict_n += len(values) > 1
            changed_n += any(r[field] != snapshots[key][field] for r in history)
        raw_rows.append(dict(field=field, flagged_loans_with_nonblank_conflict=int(conflict_n),
                             changed_from_earliest=int(changed_n)))
    raw_conflicts = pd.DataFrame(raw_rows)
    for index in fannie.index[fannie.loan_id.astype(str).isin(flagged)]:
        key = str(fannie.at[index, 'loan_id'])
        count = sum(len({r[c] for r in histories[key] if r[c]}) > 1 for c in fields)
        require(count == fannie.at[index, 'origination_field_conflict_count'], 'Conflict count does not reproduce Notebook 15.')
    numeric_cltv = fannie.ORIGINAL_CLTV_EDA
    raw_snapshot_quality = pd.DataFrame([
        ('CLTV_below_LTV', int((numeric_cltv < fannie.ORIGINAL_LTV).sum())),
        ('CLTV_missing_or_invalid', int(numeric_cltv.isna().sum())),
        ('earliest_snapshots_verified', len(raw_first))], columns=['check', 'loans'])
    raw_audit_done = True
    display(raw_conflicts)
    display(raw_snapshot_quality)
    del snapshots, histories, raw_first, parts
else:
    print('Optional raw audit not run: raw CLTV and field-level conflict investigation remain open.')


## 4. Define the population views and analysis fields

The eligible comparison uses Freddie development, Freddie 2018 Q1 and Fannie
primary eligibility. Strict Fannie eligibility is a sensitivity view; Freddie
strict eligibility has not been reconstructed, so it is not an equivalent
strict-versus-strict cross-agency comparison. The origination view includes all
available sampled loans without conditioning on target eligibility.

`ORIGINAL_UPB_EDA` is the inverse of Fannie's saved log1p balance (minor floating
point roundoff is possible). Raw CLTV is available for Fannie only after the
optional audit; it is never manufactured by adding the clipped gap to LTV.


In [ ]:
f_dev = full_freddie['SOURCE YEAR'].between(2015, 2017)
f_q1 = full_freddie['ORIGINATION COHORT'].eq('2018Q1')
ANALYSIS = SENSITIVITY + ['ORIGINAL_UPB_EDA', 'ORIGINAL_CLTV_EDA']
populations = {
    'Freddie development eligible': full_freddie.loc[f_dev & full_freddie.eligible, ANALYSIS].copy(),
    'Freddie 2018Q1 eligible': full_freddie.loc[f_q1 & full_freddie.eligible, ANALYSIS].copy(),
    'Fannie 2018Q1 primary': fannie.loc[fannie.primary, ANALYSIS].copy(),
    'Fannie 2018Q1 strict': fannie.loc[fannie.strict, ANALYSIS].copy(),
    'Freddie development sampled': full_freddie.loc[f_dev, ANALYSIS].copy(),
    'Freddie 2018Q1 sampled': full_freddie.loc[f_q1, ANALYSIS].copy(),
    'Fannie 2018Q1 available': fannie[ANALYSIS].copy(),
}
ELIGIBLE = list(populations)[:3]
ORIGINATION = list(populations)[4:]
F_PRIMARY, F_STRICT = 'Fannie 2018Q1 primary', 'Fannie 2018Q1 strict'
CONTINUOUS = ['FICO_SCORE', 'ORIGINAL_DTI', 'ORIGINAL_LTV', 'ORIGINAL_CLTV_EDA',
              'ORIGINAL_UPB_EDA', 'LOG_ORIGINAL_UPB', 'ORIGINAL_INTEREST_RATE', 'MI_PERCENT', 'CLTV_MINUS_LTV']
BINARY = [c for c in features_module.NUMERIC_FEATURES if c in SENSITIVITY
          and c not in features_module.NUMERIC_FEATURES[:7]]
CATEGORICAL = [c for c in PRIMARY if c not in features_module.NUMERIC_FEATURES]
DISPLAY_NAMES = {'FICO_SCORE': 'Credit score (Fannie sensitivity proxy)', 'ORIGINAL_DTI': 'DTI (%)',
    'ORIGINAL_LTV': 'Original LTV (%)', 'ORIGINAL_CLTV_EDA': 'Original CLTV (%)',
    'ORIGINAL_UPB_EDA': 'Original balance ($)', 'LOG_ORIGINAL_UPB': 'log1p(original balance)',
    'ORIGINAL_INTEREST_RATE': 'Original interest rate (%)', 'MI_PERCENT': 'Mortgage insurance (%)',
    'CLTV_MINUS_LTV': 'CLTV minus LTV (percentage points)'}
for name, frame in populations.items():
    require(not frame.empty, f'Empty population: {name}')
    require(not set(LABELS + target_columns).intersection(frame.columns), 'Outcome entered EDA predictors.')
    for c in CONTINUOUS:
        frame[c] = pd.to_numeric(frame[c], errors='raise')
        require(np.isfinite(frame[c].dropna()).all(), f'Nonfinite predictor: {name}/{c}')
    for c in BINARY:
        require(frame[c].notna().all() and frame[c].isin([0, 1]).all(), f'Invalid binary predictor: {name}/{c}')

population_counts = pd.DataFrame([dict(population=name, loans=len(frame)) for name, frame in populations.items()])
require(len(populations[ELIGIBLE[0]]) == EXPECTED_FREDDIE_DEVELOPMENT, 'Wrong reference population.')
numeric_availability = pd.DataFrame([
    dict(population=name, feature=c,
         status='not_retained_in_notebook15_export' if c == 'ORIGINAL_CLTV_EDA' and name.startswith('Fannie')
                and not raw_audit_done else 'available_before_imputation',
         interpretation='sensitivity_proxy_only' if c == 'FICO_SCORE' and name.startswith('Fannie') else 'EDA')
    for name in populations for c in CONTINUOUS])
display(population_counts)

OUT = FANNIE / 'notebook_15b_eda_population_comparison' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT.mkdir(parents=True, exist_ok=False)
FIGURES = OUT / 'figures'
FIGURES.mkdir()
tables = {'population_counts': population_counts, 'numeric_availability': numeric_availability,
          'target_count_reconciliation': target_summary, 'quality_membership': quality_membership,
          'strict_exclusion_overlap': eligibility_overlap, 'eligibility_reasons': eligibility_reasons,
          'conflict_distribution': conflict_distribution, 'raw_conflict_fields': raw_conflicts,
          'raw_snapshot_quality': raw_snapshot_quality, 'freddie_unmapped_purpose': purpose_unmapped,
          'earliest_snapshot_timing': snapshot_lag_summary}
tables['feature_inventory'] = pd.DataFrame([dict(feature=c,
    role='primary_candidate' if c in PRIMARY else ('fico_sensitivity_only' if c in SENSITIVITY else 'EDA_only'),
    type='continuous' if c in CONTINUOUS else ('binary' if c in BINARY else 'categorical')) for c in ANALYSIS])
figure_files = []
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})

def save_figure(fig, name):
    fig.tight_layout()
    path = FIGURES / f'{name}.png'
    fig.savefig(path, dpi=160, bbox_inches='tight')
    figure_files.append(path)
    plt.show()
    plt.close(fig)

def available(name, feature):
    return not (feature == 'ORIGINAL_CLTV_EDA' and name.startswith('Fannie') and not raw_audit_done)

print('New output folder:', OUT)


## 5. Numerical EDA: distributions, percentiles and unusual values

Tables use every observed value and preserve missing values. Outliers outside
1.5 × IQR fences are descriptive flags, not invalid records or exclusion rules.
Zero-IQR variables are reported separately. The zoomed histogram uses the
Fannie primary 1st–99th percentiles only to make the centre readable; the full
range plot, box plot and exported tables retain the tails. These plotting
limits never determine PSI bins or feature specifications.


In [ ]:
statistics, outlier_rows, missing_rows = [], [], []
for name, frame in populations.items():
    for c in ANALYSIS:
        measured = available(name, c)
        missing_rows.append(dict(population=name, feature=c, loans=len(frame),
            missing=int(frame[c].isna().sum()) if measured else np.nan,
            missing_percent=100*frame[c].isna().mean() if measured else np.nan,
            distinct=int(frame[c].nunique()) if measured else np.nan,
            constant_nonmissing=bool(frame[c].nunique() == 1) if measured else False,
            status='measured' if measured else 'source_not_retained'))
    for c in CONTINUOUS:
        if not available(name, c):
            continue
        v = frame[c].dropna()
        row = dict(population=name, feature=c, loans=len(frame), observed=len(v), missing=len(frame)-len(v))
        row.update(v.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_dict())
        row['skewness'] = v.skew() if len(v) >= 3 else np.nan
        statistics.append(row)
        q1, q3 = (v.quantile(.25), v.quantile(.75)) if len(v) else (np.nan, np.nan)
        iqr = q3-q1
        lo, hi = q1-1.5*iqr, q3+1.5*iqr
        flagged = v.lt(lo) | v.gt(hi)
        outlier_rows.append(dict(population=name, feature=c, observed=len(v), iqr=iqr,
            lower_fence=lo, upper_fence=hi, outside_fences=int(flagged.sum()),
            outside_percent=100*flagged.mean() if len(v) else np.nan,
            status='no_observations' if v.empty else ('zero_iqr_descriptive_only' if iqr == 0 else 'descriptive_only')))
numeric_summary = pd.DataFrame(statistics)
missingness = pd.DataFrame(missing_rows)
context_rows = []
for name, frame in populations.items():
    checks = {
        'FICO_outside_300_850': frame.FICO_SCORE.notna() & ~frame.FICO_SCORE.between(300, 850),
        'DTI_above_65_reference_context_only': frame.ORIGINAL_DTI.gt(65),
        'LTV_above_100_leverage_context_only': frame.ORIGINAL_LTV.gt(100),
        'nonpositive_balance': frame.ORIGINAL_UPB_EDA.le(0),
        'nonpositive_interest_rate': frame.ORIGINAL_INTEREST_RATE.le(0),
        'MI_outside_0_55': frame.MI_PERCENT.notna() & ~frame.MI_PERCENT.between(0, 55),
        'negative_clipped_gap': frame.CLTV_MINUS_LTV.lt(0),
    }
    for check, mask in checks.items():
        context_rows.append(dict(population=name, check=check, flagged_loans=int(mask.sum()),
            percent_of_population=100*mask.mean(), action='review_only_no_exclusion'))
    measured = available(name, 'ORIGINAL_CLTV_EDA')
    mask = frame.ORIGINAL_CLTV_EDA.lt(frame.ORIGINAL_LTV)
    context_rows.append(dict(population=name, check='raw_CLTV_below_LTV',
        flagged_loans=int(mask.sum()) if measured else np.nan,
        percent_of_population=100*mask.mean() if measured else np.nan,
        action='review_only_no_exclusion' if measured else 'source_not_retained'))
tables['numeric_quality_and_context'] = pd.DataFrame(context_rows)
tables.update(numeric_descriptive_statistics=numeric_summary, outlier_summary=pd.DataFrame(outlier_rows),
              missingness_and_constants=missingness)
display(numeric_summary.loc[numeric_summary.population.eq(F_PRIMARY)].round(4))
display(tables['outlier_summary'].loc[lambda d: d.population.eq(F_PRIMARY)].round(3))

for c in CONTINUOUS:
    if not available(F_PRIMARY, c):
        print('Fannie raw CLTV: skipped; enable the optional audit to retain the true source values.')
        continue
    v = populations[F_PRIMARY][c].dropna().to_numpy(dtype=float)
    if not len(v):
        continue
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
    axes[0].hist(v, bins=40, color='#287b8e')
    axes[0].set_title('Full observed range')
    lo, hi = np.quantile(v, [.01, .99])
    zoom = v[(v >= lo) & (v <= hi)]
    axes[1].hist(zoom, bins=35, color='#287b8e')
    axes[1].set_title(f'Central 98% view ({len(zoom)/len(v):.1%} retained)')
    axes[2].boxplot(v, vert=False, flierprops={'markersize': 1, 'alpha': .2})
    axes[2].set_title('Box plot, all observations')
    for ax in axes:
        ax.set_xlabel(DISPLAY_NAMES[c])
    fig.suptitle(f'Fannie primary: {DISPLAY_NAMES[c]} | observed n={len(v):,}', y=1.05)
    save_figure(fig, f'fannie_numeric_{c.lower()}')

for names, suffix in [(ELIGIBLE, 'eligible'), (ORIGINATION, 'originations')]:
    pivot = (missingness.loc[missingness.population.isin(names)]
             .pivot(index='feature', columns='population', values='missing_percent').reindex(columns=names))
    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(pivot.to_numpy(), aspect='auto', vmin=0, vmax=100, cmap='YlOrRd')
    ax.set_xticks(range(len(names)), names, rotation=15, ha='right')
    ax.set_yticks(range(len(pivot)), pivot.index)
    fig.colorbar(im, ax=ax, label='Missing (%) | blank = source unavailable')
    ax.set_title('Missingness before imputation')
    save_figure(fig, f'missingness_{suffix}')


## 6. Categorical EDA and binary indicators

Frequency tables retain all categories and include missing values in the
denominator. Plots show at most 12 observed levels plus an explicit remainder;
state and loan-term tables remain complete. Tagged values keep a real category
named MISSING distinct from an absent value. Binary flags preserve the original
contract: a false insurance/subordinate-financing flag can coexist with a
missing input. Inspect the separate missingness indicators alongside these flags.


In [ ]:
def category_tokens(series):
    return ('V:' + series.astype('string').str.strip()).fillna('M:')

def token_label(token):
    return '[Missing]' if token == 'M:' else str(token)[2:]

category_rows = []
for name, frame in populations.items():
    for c in CATEGORICAL + BINARY:
        counts = category_tokens(frame[c]).value_counts()
        for token, n in counts.items():
            category_rows.append(dict(population=name, feature=c, token=token, level=token_label(token),
                is_missing=token == 'M:', count=int(n), percent=100*n/len(frame)))
categorical_summary = pd.DataFrame(category_rows)
tables['categorical_distributions'] = categorical_summary
display(categorical_summary.loc[categorical_summary.population.eq(F_PRIMARY)])
for c in CATEGORICAL + BINARY:
    summary = categorical_summary.loc[categorical_summary.population.eq(F_PRIMARY)
                                      & categorical_summary.feature.eq(c)].sort_values('percent', ascending=False)
    shown = summary.head(12).copy()
    if len(summary) > 12:
        shown = pd.concat([shown, pd.DataFrame([dict(level='[Other levels]', percent=summary.iloc[12:].percent.sum())])])
    fig, ax = plt.subplots(figsize=(8, max(2.5, .3*len(shown))))
    ax.barh(np.arange(len(shown)), shown.percent, color='#287b8e')
    ax.set_yticks(np.arange(len(shown)), shown.level)
    ax.invert_yaxis()
    ax.set_xlabel('Share of all primary eligible loans (%)')
    ax.set_title(f'Fannie primary: {c}')
    save_figure(fig, f'fannie_categorical_{c.lower()}')


## 7. Correlations and direct population comparisons

Pearson measures linear association; Spearman measures rank association.
Correlations use pairwise observed values, and pairwise sample sizes are exported.
Constants have undefined correlations. Derived-feature relationships are partly
mechanical: balance/log balance and LTV/CLTV/gap correlations do not establish
causes of default. We do not inspect correlations with the target.

Comparative histograms use common edges and per-population probability mass,
so the larger Fannie population does not dominate the plot. The same-period
comparison reduces time differences but cannot isolate an agency effect from
sampling, acquisition coverage or product mix.


In [ ]:
correlation_rows, pair_rows = [], []
for name, frame in populations.items():
    cols = [c for c in CONTINUOUS if available(name, c)]
    pairs = frame[cols].notna().astype('int64').T.dot(frame[cols].notna().astype('int64'))
    for a in cols:
        for b in cols:
            pair_rows.append(dict(population=name, feature_a=a, feature_b=b, observed_pairs=int(pairs.loc[a, b])))
    for method in ['pearson', 'spearman']:
        corr = frame[cols].corr(method=method, min_periods=3)
        for a in cols:
            for b in cols:
                correlation_rows.append(dict(population=name, method=method, feature_a=a,
                                              feature_b=b, correlation=corr.loc[a, b]))
        if name == F_PRIMARY:
            fig, ax = plt.subplots(figsize=(9, 8))
            im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
            ax.set_xticks(range(len(cols)), cols, rotation=70, ha='right')
            ax.set_yticks(range(len(cols)), cols)
            for i in range(len(cols)):
                for j in range(len(cols)):
                    value = corr.iloc[i, j]
                    ax.text(j, i, f'{value:.2f}' if pd.notna(value) else 'NA', ha='center', va='center', fontsize=8)
            fig.colorbar(im, ax=ax, label=method.capitalize() + ' correlation')
            ax.set_title(f'Fannie primary: {method.capitalize()} correlation')
            save_figure(fig, f'fannie_correlation_{method}')
tables['predictor_correlations'] = pd.DataFrame(correlation_rows)
tables['correlation_pair_counts'] = pd.DataFrame(pair_rows)

for names, suffix in [(ELIGIBLE, 'eligible'), (ORIGINATION, 'originations')]:
    for c in CONTINUOUS:
        included = [name for name in names if available(name, c) and populations[name][c].notna().any()]
        if len(included) < 2:
            continue
        values = [populations[name][c].dropna().to_numpy(dtype=float) for name in included]
        edges = np.histogram_bin_edges(np.concatenate(values), bins=40)
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
        for name, v in zip(included, values):
            axes[0].hist(v, bins=edges, weights=np.full(len(v), 100/len(v)),
                         histtype='step', linewidth=1.8, label=name)
        axes[0].set_ylabel('Observed loans per bin (%)')
        axes[0].legend(fontsize=8)
        axes[1].boxplot(values, vert=False,
                        flierprops={'markersize': 1, 'alpha': .1})
        axes[1].set_yticks(range(1, len(included)+1), included, fontsize=8)
        for ax in axes:
            ax.set_xlabel(DISPLAY_NAMES[c])
        fig.suptitle(f'Population comparison: {DISPLAY_NAMES[c]}', y=1.03)
        save_figure(fig, f'comparison_{suffix}_{c.lower()}')

    for c in CATEGORICAL:
        sub = categorical_summary.loc[categorical_summary.population.isin(names) & categorical_summary.feature.eq(c)]
        pivot = sub.pivot(index='token', columns='population', values='percent').fillna(0).reindex(columns=names)
        order = pivot.max(axis=1).sort_values(ascending=False).index
        top = pivot.loc[order[:12]].copy()
        if len(order) > 12:
            top.loc['OTHER_PLOT_ONLY'] = pivot.loc[order[12:]].sum()
        top.index = ['[Other levels]' if t == 'OTHER_PLOT_ONLY' else token_label(t) for t in top.index]
        fig, ax = plt.subplots(figsize=(11, max(3, .45*len(top))))
        top.plot.barh(ax=ax, width=.8)
        ax.set_xlabel('Share of population (%)')
        ax.set_ylabel('')
        ax.set_title(f'Population comparison: {c}')
        ax.legend(fontsize=8)
        save_figure(fig, f'comparison_{suffix}_{c.lower()}')

policy_comparison = (numeric_summary.loc[numeric_summary.population.isin([F_PRIMARY, F_STRICT])]
    .pivot(index='feature', columns='population', values=['mean', '50%', 'std']))
tables['primary_strict_numeric_comparison'] = policy_comparison.reset_index().set_axis(
    ['feature'] + [f'{a} | {b}' for a, b in policy_comparison.columns], axis=1)
display(tables['primary_strict_numeric_comparison'].round(4))


## 8. Quantify distribution differences

Numeric PSI uses the same pooled Freddie development deciles, duplicate-cut
removal, right-closed bins and count smoothing as Notebook 12 (0.5 primary;
0.1 and 1.0 sensitivity). Missingness is separate. Eligible and sampled views
have their own Freddie development reference. Those development bins also stay
fixed for the Freddie 2018 Q1–Fannie comparison: Fannie never defines the bins.

Wasserstein is in the feature's units (dollars, score points or percentage
points); development-IQR scaling is unitless. A zero reference IQR is reported
as unavailable for scaling. Constant references use percent changed from the
constant rather than a misleading numeric PSI of zero.

Categorical distance is total variation, 0–1, including missing values. Binary
and missingness distributions also report smoothed two-bin PSI and signed
percentage-point changes. This is a descriptive extension alongside Notebook
12's original binary measure, not a replacement for earlier results. No
distance threshold is treated as evidence of model deterioration.


In [ ]:
SMOOTHING = [.1, .5, 1.0]

def observed(series):
    v = pd.to_numeric(series, errors='raise').dropna().to_numpy(dtype=float)
    require(np.isfinite(v).all(), 'Nonfinite numeric distribution.')
    return v

def bin_counts(values, cuts):
    return np.bincount(np.searchsorted(cuts, values, side='left'), minlength=len(cuts)+1)

def psi_from_counts(reference, current, addition=.5):
    r, c = np.asarray(reference, dtype=float), np.asarray(current, dtype=float)
    require(r.shape == c.shape and r.sum() > 0 and c.sum() > 0 and addition > 0, 'Invalid PSI counts.')
    r = (r+addition)/(r.sum()+addition*len(r))
    c = (c+addition)/(c.sum()+addition*len(c))
    return float(np.sum((c-r)*np.log(c/r)))

COMPARISONS = []
for view, names in [('eligible', ELIGIBLE), ('originations', ORIGINATION)]:
    dev_name, matched_name, fannie_name = names
    for ref_name, cur_name in [(dev_name, matched_name), (dev_name, fannie_name), (matched_name, fannie_name)]:
        COMPARISONS.append((view, dev_name, ref_name, cur_name))
COMPARISONS += [('eligible_strict_sensitivity', ELIGIBLE[0], ELIGIBLE[0], F_STRICT),
                ('eligibility_sensitivity', ELIGIBLE[0], F_PRIMARY, F_STRICT)]
distance_rows, bin_rows, smoothing_rows, support_rows, reference_rows = [], [], [], [], []
for view, dev_name, ref_name, cur_name in COMPARISONS:
    reference, current, development = populations[ref_name], populations[cur_name], populations[dev_name]
    for c in CONTINUOUS:
        row = dict(view=view, bin_reference=dev_name, reference=ref_name, comparison=cur_name,
                   feature=c, reference_loans=len(reference), comparison_loans=len(current))
        if not all(available(name, c) for name in [ref_name, cur_name, dev_name]):
            distance_rows.append(dict(row, status='source_not_retained'))
            continue
        d, r, v = observed(development[c]), observed(reference[c]), observed(current[c])
        row.update(reference_observed=len(r), comparison_observed=len(v),
                   missing_change_pp=100*(current[c].isna().mean()-reference[c].isna().mean()))
        if not len(d) or not len(r) or not len(v):
            distance_rows.append(dict(row, status='no_observations'))
            continue
        iqr = float(np.quantile(d, .75)-np.quantile(d, .25))
        row.update(wasserstein_raw=float(wasserstein_distance(r, v)), development_iqr=iqr,
                   wasserstein_scaled_status='ok' if iqr > 0 else 'development_iqr_zero')
        row['wasserstein_iqr_scaled'] = row['wasserstein_raw']/iqr if iqr > 0 else np.nan
        if np.unique(d).size == 1 or np.unique(r).size == 1:
            constant = float(r[0]) if np.unique(r).size == 1 else float(d[0])
            row.update(status='constant_reference', psi=np.nan,
                       constant_value=constant, reference_changed_percent=100*np.mean(r != constant),
                       comparison_changed_percent=100*np.mean(v != constant))
        else:
            cuts = np.unique(np.quantile(d, np.arange(.1, 1., .1), method='linear'))
            rc, cc = bin_counts(r, cuts), bin_counts(v, cuts)
            require(rc.sum() == len(r) and cc.sum() == len(v), 'Numeric bins lost observations.')
            row.update(status='ok', psi=psi_from_counts(rc, cc), bins=len(rc))
            for alpha in SMOOTHING:
                smoothing_rows.append(dict(row, smoothing_count=alpha, psi=psi_from_counts(rc, cc, alpha)))
            boundaries = [-np.inf, *cuts.tolist(), np.inf]
            for j, (a, b) in enumerate(zip(rc, cc)):
                bin_rows.append(dict(view=view, reference=ref_name, comparison=cur_name, feature=c, bin=j+1,
                    lower=boundaries[j], upper=boundaries[j+1], reference_count=int(a), comparison_count=int(b),
                    reference_percent=100*a/len(r), comparison_percent=100*b/len(v)))
        distance_rows.append(row)
    for c in CATEGORICAL + BINARY:
        rc = category_tokens(reference[c]).value_counts().to_dict()
        cc = category_tokens(current[c]).value_counts().to_dict()
        support = sorted(set(rc) | set(cc) | {'M:'})
        tv = .5*sum(abs(rc.get(t, 0)/len(reference)-cc.get(t, 0)/len(current)) for t in support)
        distance_rows.append(dict(view=view, bin_reference=dev_name, reference=ref_name, comparison=cur_name,
            feature=c, status='categorical' if c in CATEGORICAL else 'binary', total_variation=tv,
            unseen_percent=100*sum(cc.get(t, 0) for t in cc if t != 'M:' and t not in rc)/len(current),
            signed_binary_change_pp=100*(current[c].mean()-reference[c].mean()) if c in BINARY else np.nan))
        for t in support:
            support_rows.append(dict(view=view, reference=ref_name, comparison=cur_name, feature=c,
                token=t, level=token_label(t), reference_count=rc.get(t, 0), comparison_count=cc.get(t, 0),
                signed_change_pp=100*(cc.get(t, 0)/len(current)-rc.get(t, 0)/len(reference))))
        if c in BINARY:
            counts_r = [int(reference[c].eq(0).sum()), int(reference[c].eq(1).sum())]
            counts_c = [int(current[c].eq(0).sum()), int(current[c].eq(1).sum())]
            for alpha in SMOOTHING:
                smoothing_rows.append(dict(view=view, reference=ref_name, comparison=cur_name, feature=c,
                    representation='binary', smoothing_count=alpha, psi=psi_from_counts(counts_r, counts_c, alpha)))
    for c in SENSITIVITY:
        mr, mc = int(reference[c].isna().sum()), int(current[c].isna().sum())
        for alpha in SMOOTHING:
            smoothing_rows.append(dict(view=view, reference=ref_name, comparison=cur_name, feature=c,
                representation='missingness', smoothing_count=alpha,
                psi=psi_from_counts([len(reference)-mr, mr], [len(current)-mc, mc], alpha),
                signed_change_pp=100*(mc/len(current)-mr/len(reference))))

# Save one independent record of development bins and scaling for reproducibility.
for name in [ELIGIBLE[0], ORIGINATION[0]]:
    for c in CONTINUOUS:
        v = observed(populations[name][c])
        reference_rows.append(dict(population=name, feature=c, observed=len(v),
            cuts=np.unique(np.quantile(v, np.arange(.1, 1., .1))).tolist() if len(v) and np.unique(v).size > 1 else [],
            constant=float(v[0]) if len(v) and np.unique(v).size == 1 else None,
            iqr=float(np.quantile(v, .75)-np.quantile(v, .25)) if len(v) else None))
distances = pd.DataFrame(distance_rows)
tables.update(distribution_distances=distances, numeric_bin_contributions=pd.DataFrame(bin_rows),
              psi_smoothing_sensitivity=pd.DataFrame(smoothing_rows), category_changes=pd.DataFrame(support_rows))
display(distances.loc[distances['view'].eq('eligible')].round(5))

for metric, features, label in [('psi', CONTINUOUS, 'Numeric PSI (0.5 count smoothing)'),
        ('wasserstein_iqr_scaled', CONTINUOUS, 'Wasserstein / development IQR'),
        ('total_variation', CATEGORICAL, 'Total variation (0–1)')]:
    sub = distances.loc[distances['view'].eq('eligible') & distances.feature.isin(features)].copy()
    sub['pair'] = sub.reference + ' → ' + sub.comparison
    pivot = sub.pivot(index='feature', columns='pair', values=metric)
    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.plot.barh(ax=ax)
    ax.set_xlabel(label)
    ax.set_ylabel('')
    ax.set_title('Distribution differences | unavailable comparisons left blank')
    ax.legend(fontsize=7)
    save_figure(fig, f'distribution_{metric}')


## 9. Compatibility and readiness for review

This notebook describes population differences; it cannot establish semantic
equivalence, identify causal agency effects or demonstrate model deterioration.
The original 29-predictor models remain untouched. The 22/24 candidate contracts
remain provisional. FICO differences, acquisition coverage, earliest-snapshot
limitations and imperfect adverse-terminal-event equivalence remain material.

The optional raw audit identifies changes but cannot prove that a field was
published or available at origination. Review release-specific documentation
before approving companion development or scoring. A written note below is
evidence for review, not automatic approval.


In [ ]:
missing_primary = missingness.loc[missingness.population.eq(F_PRIMARY)
                                 & missingness.feature.isin(SENSITIVITY)]
constant_primary = missing_primary.loc[missing_primary.constant_nonmissing, 'feature'].tolist()
all_missing_primary = missing_primary.loc[missing_primary.missing_percent.eq(100), 'feature'].tolist()
readiness = pd.DataFrame([
    ('notebook15_counts_and_fingerprints', 'passed', 'Candidate counts, hashes and review ZIP reconciled.'),
    ('strict_exclusion_reconciliation', 'passed', 'Exact joint criteria and overlap exported; no rule changes.'),
    ('origination_field_conflicts', 'evidence_for_review' if raw_audit_done else 'pending_raw_investigation',
     f'{EXPECTED_N15["conflicts"]} loans flagged; none silently excluded or replaced.'),
    ('raw_cltv_eda', 'available' if raw_audit_done else 'unavailable_in_processed_export',
     'No reconstruction from LTV plus clipped gap.'),
    ('earliest_snapshot_availability', 'unresolved', 'Earliest report is not proof of origination-time publication.'),
    ('release_provenance', 'evidence_for_review' if RELEASE_REVIEW_NOTE.strip() else 'pending_review',
     RELEASE_REVIEW_NOTE.strip() or meta.get('release_evidence') or 'Match source release to its glossary.'),
    ('cross_agency_target_semantics', 'evidence_for_review' if TARGET_REVIEW_NOTE.strip() else 'pending_review',
     TARGET_REVIEW_NOTE.strip() or 'Composite adverse-event mapping remains imperfect; numeric-only outcomes need compatible training.'),
    ('fico_equivalence', 'sensitivity_only', 'Minimum disclosed-score proxy is not certified equivalent to Freddie.'),
    ('purchase_refinance_harmonization', 'pending_review' if len(purpose_unmapped) else 'implemented_candidate',
     f'{int(purpose_unmapped.loans.sum()) if len(purpose_unmapped) else 0} unmapped Freddie loans; review before freezing.'),
    ('constant_primary_features', 'descriptive', ', '.join(constant_primary) or 'None'),
    ('all_missing_primary_features', 'pending_review' if all_missing_primary else 'none', ', '.join(all_missing_primary) or 'None'),
    ('population_coverage_and_product_mix', 'documented_limitation',
     'One Fannie acquisition file versus Freddie annual samples; no complete origination cohort or matched product restriction asserted.'),
    ('companion_model_development', 'requires_review', 'Review 15B findings and freeze the common specification before Notebook 16.'),
    ('model_scoring', 'blocked', 'No models loaded, fitted or scored; external scoring requires compatibility review.')
], columns=['check', 'status', 'evidence_or_limitation'])
tables['readiness_assessment'] = readiness
display(readiness)


## 10. Save aggregate results and the review bundle

This run creates a new timestamped folder. It saves no identifiers, individual
feature rows, labels, predictions or models. The review ZIP includes CSV tables,
PNG figures, reference bins, provenance and a short data-driven review note.

After the cell finishes, save this executed notebook in VS Code and share
`notebook_15b_eda_population_review.zip` for interpretation and milestone review.
Do not begin Notebook 16 until these findings and outstanding issues are reviewed.


In [ ]:
for name, path in paths.items():
    require(signatures[name] == (path.stat().st_size, path.stat().st_mtime_ns), f'Input changed during EDA: {name}')
for name, table in tables.items():
    table.to_csv(OUT / f'{name}.csv', index=False)
reference_payload = dict(source='Freddie 2015–2017, before imputation',
    quantiles=list(np.arange(.1, 1., .1)), quantile_method='linear',
    boundary_convention='searchsorted side=left; right-closed interior bins; unbounded tails',
    primary_smoothing=.5, sensitivity_smoothing=SMOOTHING,
    missingness='separate for numeric; explicit tagged category for categorical TV',
    reference_bins=reference_rows)
(OUT / 'development_reference_bins.json').write_text(json.dumps(reference_payload, indent=2, allow_nan=False), encoding='utf-8')

high_missing = missing_primary.loc[missing_primary.missing_percent.gt(0), ['feature', 'missing_percent']]
notes = [
    '# Notebook 15B — EDA review', '',
    f'Fannie actual 2018 Q1 originations in the available acquisition file: {len(fannie):,}.',
    f'Primary eligible: {int(fannie.primary.sum()):,}; strict: {int(fannie.strict.sum()):,}.',
    f'Additional strict exclusions: {int(additional.sum()):,}; overlap is in strict_exclusion_overlap.csv.',
    f'Raw audit completed: {raw_audit_done}.',
    f'Primary constant features: {", ".join(constant_primary) or "None"}.',
    f'Primary all-missing features: {", ".join(all_missing_primary) or "None"}.',
    '', 'Primary predictor missingness (%):', high_missing.to_string(index=False),
    '', 'Review the numerical and categorical tables alongside the saved plots.',
    'Distribution distance alone does not demonstrate loss of model reliability.',
    'All outlier flags are descriptive; no observation or predictor was removed.',
    'No target-conditioned EDA, feature selection, training or scoring was performed.',
    'Review readiness_assessment.csv before progressing to Notebook 16.',
    'One acquisition-file subset cannot validate the complete Freddie temporal study.',
]
(OUT / 'review_notes.md').write_text('\n'.join(notes) + '\n', encoding='utf-8')
provenance = dict(notebook='15b_fannie_eda_population_comparison.ipynb',
    source_commit=SOURCE_COMMIT, created_utc=datetime.now(timezone.utc).isoformat(),
    input_paths={k: str(v) for k, v in paths.items()}, input_sha256=hashes,
    primary_predictors=PRIMARY, sensitivity_predictors=SENSITIVITY,
    raw_audit_completed=raw_audit_done, raw_source_sha256=meta['source_sha256'] if raw_audit_done else None,
    raw_cltv_available=raw_audit_done, release_review_note=RELEASE_REVIEW_NOTE,
    target_review_note=TARGET_REVIEW_NOTE, scope=meta['scope'],
    comparisons=[dict(view=v, bin_reference=d, reference=r, comparison=c) for v, d, r, c in COMPARISONS],
    models_fitted=False, models_scored=False, final_compatibility_approved=False,
    input_validation='passed', dataset_execution='completed',
    versions=dict(python=platform.python_version(), pandas=pd.__version__, numpy=np.__version__,
                  scipy=scipy.__version__, matplotlib=matplotlib.__version__))
(OUT / 'metadata.json').write_text(json.dumps(provenance, indent=2, allow_nan=False), encoding='utf-8')
artifacts = sorted(p for p in OUT.rglob('*') if p.is_file())
inventory = pd.DataFrame([dict(file=str(p.relative_to(OUT)), bytes=p.stat().st_size, sha256=sha256(p)) for p in artifacts])
inventory.to_csv(OUT / 'output_inventory.csv', index=False)
review_zip = OUT / 'notebook_15b_eda_population_review.zip'
with zipfile.ZipFile(review_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p != review_zip:
            require(p.suffix in {'.csv', '.json', '.md', '.png'}, 'Unexpected review artifact.')
            archive.write(p, p.relative_to(OUT))
with zipfile.ZipFile(review_zip) as archive:
    require(archive.testzip() is None, 'Review ZIP integrity check failed.')
    require('readiness_assessment.csv' in archive.namelist() and len(figure_files) > 0, 'Review bundle incomplete.')
print(f'Saved {len(tables)} aggregate tables and {len(figure_files)} analysis figures.')
print('Output folder:', OUT)
print('Share this aggregate review ZIP:', review_zip)
print('Notebook 15B execution finished. Compatibility approval and external scoring remain pending.')
